In [17]:
import numpy as np
import cv2
# import imutils
import os
from os.path import join
import math
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

In [18]:
DATASET = './evs_mot_public_dataset'
TRAIN = os.path.join(DATASET, "evs_mot-train")
SEQUENCE = "MOT_04"

img = os.path.join(TRAIN, SEQUENCE, 'img1/%06d.jpg')
det_file = os.path.join(TRAIN, SEQUENCE, 'det', 'det.txt')
gt_file = os.path.join(TRAIN, SEQUENCE, 'gt', 'gt.txt')

In [20]:
def calculate_iou(boxA,boxB):
    boxA_x1, boxA_y1 = boxA[0], boxA[1]
    boxA_x2, boxA_y2 = boxA[0] + boxA[2], boxA[1] + boxA[3]
    boxB_x1, boxB_y1 = boxB[0], boxB[1]
    boxB_x2, boxB_y2 = boxB[0] + boxB[2], boxB[1] + boxB[3]

    xA = max(boxA_x1, boxB_x1)
    yA = max(boxA_y1, boxB_y1)
    xB = min(boxA_x2, boxB_x2)
    yB = min(boxA_y2, boxB_y2)

    width_iou = max(0, xB - xA)
    height_iou = max(0, yB - yA)
    interArea = width_iou * height_iou
    boxAreaA = boxA[2] * boxA[3]
    boxAreaB = boxB[2] * boxB[3]
    iou = interArea / (boxAreaA + boxAreaB - interArea)
    return iou

def calculate_distance(boxA, boxB):
    cx_A = boxA[0] + boxA[2] / 2
    cy_A = boxA[1] + boxA[3] / 2

    cx_B = boxB[0] + boxB[2] / 2
    cy_B = boxB[1] + boxB[3] / 2
    
    distance = ((cx_A - cx_B)**2 + (cy_A - cy_B)**2)**0.5
    
    return distance

In [21]:
# ===== KALMAN TRACK CLASS =====
class KalmanTrack:
    def __init__(self, track_id, bbox):
        self.id = track_id
        self.bbox = np.array(bbox, dtype=np.float32)
        self.velocity = np.array([0.0, 0.0, 0.0, 0.0], dtype=np.float32)
        self.lost = 0
    
    def predict(self):
        return self.bbox + self.velocity
    
    def update(self, bbox):
        bbox = np.array(bbox, dtype=np.float32)
        self.velocity = (bbox - self.bbox) * 0.5 + self.velocity * 0.5
        self.bbox = bbox
        self.lost = 0

# ===== HUNGARIAN COST MATRIX =====
def compute_cost_matrix(active_tracks, detections, frame_width, frame_height):
    if not active_tracks or not detections:
        return np.array([]), list(active_tracks.keys())
    
    track_ids = list(active_tracks.keys())
    cost_matrix = np.zeros((len(track_ids), len(detections)))
    
    for i, track_id in enumerate(track_ids):
        predicted_bbox = active_tracks[track_id].predict()
        for j, det in enumerate(detections):
            iou = calculate_iou(predicted_bbox, det)
            dist = calculate_distance(predicted_bbox, det)
            norm_dist = dist / np.sqrt(frame_width**2 + frame_height**2)
            cost_matrix[i, j] = -0.7 * iou + 0.3 * norm_dist
    
    return cost_matrix, track_ids

raw_det = np.loadtxt(det_file, delimiter=',')
detections_dict = {}

for row in raw_det:
    f = int(row[0])
    conf = row[6]
    if conf < 0.4:
        continue
    if f not in detections_dict:
        detections_dict[f] = []
    detections_dict[f].append(row[2:6])

# ===== TRACKING =====
active_tracks = {}
next_id = 1
lost_nr = 25
cap = cv2.VideoCapture(img, cv2.CAP_IMAGES)
frame_nr = 1
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

while True:
    ret, frame = cap.read()
    if not ret:
        break

    current_det = detections_dict.get(frame_nr, [])
    
    # ===== HUNGARIAN MATCHING =====
    if active_tracks and current_det:
        cost_matrix, track_ids = compute_cost_matrix(active_tracks, current_det, frame_width, frame_height)
        track_indices, det_indices = linear_sum_assignment(cost_matrix)
        matched_pairs = set(track_ids[i] for i in track_indices)
    else:
        track_indices, det_indices = np.array([]), np.array([])
        track_ids = []
        matched_pairs = set()
    
    # ===== TE PĘTLE MUSZĄ BYĆ NA ZEWNĄTRZ if/else =====
    new_tracks = {}
    used_dets = set()
    results_list = []

    # Update matched tracks
    for track_idx, det_idx in zip(track_indices, det_indices):
        track_id = track_ids[track_idx]
        active_tracks[track_id].update(current_det[det_idx])
        new_tracks[track_id] = active_tracks[track_id]
        used_dets.add(det_idx)
    
    # Unmatched detections = new tracks
    for det_idx, det in enumerate(current_det):
        if det_idx not in used_dets:
            new_tracks[next_id] = KalmanTrack(next_id, det)
            next_id += 1
    
    # Unmatched tracks = lost++
    for t_id, track in active_tracks.items():
        if t_id not in matched_pairs:
            track.lost += 1
            if track.lost <= lost_nr:
                new_tracks[t_id] = track
    
    active_tracks = new_tracks
    
    # DRAW
    for t_id, track in active_tracks.items():
        if track.lost == 0:
            x, y, w, h = map(int, track.bbox)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 255, 0), 2)
            cv2.putText(frame, f'ID: {t_id}', (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 2)

    cv2.imshow('Tracking', frame)
    frame_nr += 1
    if cv2.waitKey(15) & 0xFF == ord('q'): 
        break

cap.release()
cv2.destroyAllWindows()


In [22]:
raw_det = np.loadtxt(det_file, delimiter=',')
detections_dict = {}

for row in raw_det:
    f = int(row[0])
    conf = row[6]
    if conf < 0.15:
        continue
    if f not in detections_dict:
        detections_dict[f] = []
    detections_dict[f].append(row[2:6])

active_tracks = {}
next_id = 1
lost_nr = 30
cap = cv2.VideoCapture(img, cv2.CAP_IMAGES)
frame_nr = 1
results_list = []

while True:
    ret, frame = cap.read()
    if not ret: break

    current_det = detections_dict.get(frame_nr, [])
    track_ids = list(active_tracks.keys())
    
    # 1. Budujemy macierz kosztów (Im mniejszy koszt, tym lepiej)
    # Wiersze = Ślady (tracks), Kolumny = Detekcje
    if len(track_ids) > 0 and len(current_det) > 0:
        cost_matrix = np.zeros((len(track_ids), len(current_det)))
        
        for i, t_id in enumerate(track_ids):
            for j, det in enumerate(current_det):
                t_bbox = active_tracks[t_id]["bbox"]
                iou = calculate_iou(det, t_bbox)
                dist = calculate_distance(det, t_bbox)
                
                # Koszt: chcemy mały dystans i DUŻE IoU, więc 1 - IoU
                # Jeśli IoU jest zerowe, dajemy bardzo wysoki koszt
                if iou < 0.1:
                    cost_matrix[i, j] = 1000 
                else:
                    cost_matrix[i, j] = (1 - iou) * 0.7 + (dist / 500) * 0.3

        # 2. Algorytm Węgierski znajduje optymalne pary
        row_ind, col_ind = linear_sum_assignment(cost_matrix)
        
        matched_track_indices = set()
        matched_det_indices = set()

        new_tracks = {}
        for r, c in zip(row_ind, col_ind):
            # Sprawdzamy próg odcięcia (czy dopasowanie nie jest zbyt słabe)
            if cost_matrix[r, c] < 0.8: # Odpowiednik Twojego score > 0.4
                t_id = track_ids[r]
                new_tracks[t_id] = {"bbox": current_det[c], "lost": 0}
                matched_track_indices.add(r)
                matched_det_indices.add(c)

    else:
        new_tracks = {}
        matched_det_indices = set()
        matched_track_indices = set()

    # 3. Obsługa nowych detekcji (które nie zostały dopasowane)
    for j, det in enumerate(current_det):
        if j not in matched_det_indices:
            new_tracks[next_id] = {"bbox": det, "lost": 0}
            next_id += 1

    # 4. Obsługa zgubionych śladów (które nie dostały detekcji)
    for i, t_id in enumerate(track_ids):
        if i not in matched_track_indices:
            track = active_tracks[t_id]
            track["lost"] += 1
            if track["lost"] < lost_nr:
                new_tracks[t_id] = track

    active_tracks = new_tracks

    for t_id, track in active_tracks.items():
        if track["lost"] == 0:
            bbox = track["bbox"]
            x, y, w, h = bbox
            results_list.append(f"{frame_nr},{t_id},{x:.1f},{y:.1f},{w:.1f},{h:.1f},1,-1,-1,-1\n")

            x1, y1, w1, h1 = map(int, bbox)
            cv2.rectangle(frame, (x1, y1), (x1 + w1, y1 + h1), (255, 255, 0), 2)
            cv2.putText(frame, f'ID: {t_id}', (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 2)

    cv2.imshow('Tracking', frame)
    frame_nr += 1
    if cv2.waitKey(15) & 0xFF == ord('q'): 
        break

cap.release()
cv2.destroyAllWindows()